# Docker Security Scanner — Live Demo
**Student:** Swetha M  |  **Roll No:** 727823TUCY049  |  **Project:** DockerSecurityScanner

This notebook walks through all three test cases of the Docker Security Scanner pipeline.
Each cell corresponds to one stage or test case, with live output shown inline.

## Overview

The Docker Security Scanner checks:

| Check | Flag Raised When |
|---|---|
| Privileged mode | `--privileged` flag is set |
| Root user | Container user is root or unset |
| Exposed ports | Port bound to `0.0.0.0` |
| Missing memory limit | `Memory == 0` in HostConfig |
| Missing CPU quota | `CpuQuota == 0` |
| Writable root FS | `ReadonlyRootfs == False` |
| Image age | Image older than 180 days |
| Dangling image | Image has no tag |

In [ ]:
# Roll: 727823TUCY049
import subprocess, sys, json, glob, os, datetime

ROLL = '727823TUCY049'
print(f'Roll Number : {ROLL}')
print(f'Timestamp   : {datetime.datetime.now()}')

---
## Stage 1 — Setup Lab
Verifies Docker, installs dependencies, pulls `alpine:latest`, starts a deliberately misconfigured demo container.

In [ ]:
result = subprocess.run(
    [sys.executable, '../code/setup_lab.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('[STDERR]', result.stderr)

---
## Test Case 1 — Full Scan (Containers + Images, All Severities)
Demonstrates the complete scan: all container checks plus image age and tag audit.

In [ ]:
result = subprocess.run(
    [sys.executable, '../code/tool_main.py', '--mode', 'full', '--severity', 'ALL', '--tc', '1'],
    capture_output=True, text=True, cwd='../code'
)
print(result.stdout)

In [ ]:
# Parse and display TC1 findings
tc1_files = sorted(glob.glob('../code/scan_results_tc1_*.json'))
if tc1_files:
    with open(tc1_files[-1]) as f:
        data = json.load(f)
    print(f"Containers scanned : {data['summary']['containers_scanned']}")
    print(f"Container issues   : {data['summary']['container_issues']}")
    print(f"Images scanned     : {data['summary']['images_scanned']}")
    print(f"Image issues       : {data['summary']['image_issues']}")
    print()
    for c in data['container_findings']:
        print(f"  [{c['severity']}] {c['container_name']} — {c['issues']}")
else:
    print('No TC1 output file found. Run setup_lab.py first.')

---
## Test Case 2 — Containers Only, HIGH Severity Filter
Shows only critical findings (privileged mode, root user). Demonstrates severity filtering.

In [ ]:
result = subprocess.run(
    [sys.executable, '../code/tool_main.py', '--mode', 'containers', '--severity', 'HIGH', '--tc', '2'],
    capture_output=True, text=True, cwd='../code'
)
print(result.stdout)

In [ ]:
tc2_files = sorted(glob.glob('../code/scan_results_tc2_*.json'))
if tc2_files:
    with open(tc2_files[-1]) as f:
        data = json.load(f)
    high = [c for c in data['container_findings'] if c['severity'] == 'HIGH']
    print(f"HIGH-severity containers found: {len(high)}")
    for c in high:
        print(f"  Container : {c['container_name']}")
        for issue in c['issues']:
            print(f"    • {issue}")
else:
    print('No TC2 output file found.')

---
## Test Case 3 — Images Only (Age and Tag Audit)
Scans all local images for those older than 180 days or with no tag assigned.

In [ ]:
result = subprocess.run(
    [sys.executable, '../code/tool_main.py', '--mode', 'images', '--tc', '3'],
    capture_output=True, text=True, cwd='../code'
)
print(result.stdout)

In [ ]:
tc3_files = sorted(glob.glob('../code/scan_results_tc3_*.json'))
if tc3_files:
    with open(tc3_files[-1]) as f:
        data = json.load(f)
    print(f"Images scanned : {data['summary']['images_scanned']}")
    print(f"Image issues   : {data['summary']['image_issues']}")
    print()
    for img in data['image_findings']:
        if img['issues']:
            tags = ', '.join(img['tags']) if img['tags'] else '<no-tag>'
            print(f"  [{img['severity']}] {tags}")
            for issue in img['issues']:
                print(f"    • {issue}")
else:
    print('No TC3 output file found.')

---
## Stage 3 — Analyze Results
Consolidates all three scan outputs, prints severity breakdown table, and writes `consolidated_report.json`.

In [ ]:
result = subprocess.run(
    [sys.executable, '../code/analyze_results.py'],
    capture_output=True, text=True, cwd='../code'
)
print(result.stdout)

---
## Summary

The Docker Security Scanner successfully demonstrated three distinct scanning modes across three test cases. TC1 revealed misconfigured containers running in privileged mode with ports exposed on all interfaces. TC2 confirmed severity filtering works — isolating only HIGH-severity findings. TC3 audited image metadata for age and dangling tags.

**Key takeaway:** Default Docker configurations are permissive. Automated scanning is essential to catch misconfigurations before they reach production.

Roll Number: **727823TUCY049** | Student: **Swetha M**